# Banknote Authentication – Preprocessing Pipeline
SCT213-C002-0026/2023 - Caroline Nyanjui

SCT213-C002-0130/2023 - Catherine Njogu

**ML II Unsupervised Learning Capstone – Step 2**

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 2. Load Dataset

In [ ]:
data = pd.read_csv(r"C:\\Users\\hp\\Downloads\\data_banknote_authentication (1).txt",
                   header=None,
                   names=['variance', 'skewness', 'curtosis', 'entropy', 'class'])

# Print first 5 rows
data.head()

## 3. Remove Duplicates

We drop duplicate rows to ensure each observation is unique before preprocessing.

## 4. Remove Target Column

Since clustering is **unsupervised**, we drop the `class` column from `X`. It is not used during preprocessing or clustering.

> The `class` column is saved separately as `y_true` for use in external evaluation metrics (ARI, NMI) at the end of the project.

## 5. Outlier Flagging (IQR Method)

Outliers are **flagged but not removed** — this is intentional. Removing outliers in unsupervised learning can distort the natural structure of the data. Instead, an `outlier_flag` column is added to inform the clustering algorithms.

In [ ]:
data = data.drop_duplicates().reset_index(drop=True)

# Separate ground truth labels BEFORE dropping — saved for evaluation later
y_true = data['class'].copy()

# Drop class column from features
X = data.drop('class', axis=1)

# IQR Outlier Flagging
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

X['outlier_flag'] = ((X < lower_bound) | (X > upper_bound)).any(axis=1)
print(X)

## 6. Preprocessing Pipeline

We build a `sklearn` Pipeline with one step:
- **Scaler** – standardizes all features to have mean = 0 and standard deviation = 1

Standardization is critical for distance-based clustering algorithms like K-Means and DBSCAN.

> **Note:** No imputation step is included because EDA confirmed **zero missing values** in this dataset. Including an imputer when data is complete is unnecessary and adds no value.

In [ ]:
numeric_features = X.columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features)
])

## 7. PCA Dataset

We apply **Principal Component Analysis (PCA)** to reduce the feature space to 2 dimensions. This is useful for:
- Visualizing clusters in 2D
- Reducing noise in the data
- Speeding up clustering algorithms

Two datasets are produced:
- `df_no_pca` – all scaled features retained
- `df_with_pca` – reduced to PC1 and PC2 only

In [ ]:
X_no_pca = preprocessor.fit_transform(X)

df_no_pca = pd.DataFrame(
    X_no_pca,
    columns=numeric_features
)

print(df_no_pca)

In [ ]:
pca = PCA(n_components=2)
X_with_pca = pca.fit_transform(X_no_pca)

df_with_pca = pd.DataFrame(
    X_with_pca,
    columns=['PC1', 'PC2']
)

print(df_with_pca)

## 8. Explained Variance

We check how much information is retained after PCA.

In [ ]:
explained_variance = pca.explained_variance_ratio_

variance_summary = pd.DataFrame({
    'Principal_Component': ['PC1', 'PC2'],
    'Explained_Variance_Ratio': explained_variance
})

display(variance_summary)
print("Total Variance Retained:", explained_variance.sum())

## 10. Save Datasets

We save all three outputs so the clustering notebooks (K-Means, Hierarchical, DBSCAN) and the evaluation notebook can load them directly.

- `df_no_pca.csv` — scaled 5-feature dataset for clustering without PCA
- `df_with_pca.csv` — 2-component PCA dataset for clustering with PCA
- `y_true.csv` — ground truth labels for external evaluation metrics (ARI, NMI)

In [ ]:
# Save preprocessed datasets for downstream notebooks
df_no_pca.to_csv('df_no_pca.csv', index=False)
df_with_pca.to_csv('df_with_pca.csv', index=False)
y_true.to_csv('y_true.csv', index=False)

print("All datasets saved successfully.")
print("  df_no_pca.csv   →  K-Means, Hierarchical, DBSCAN (no PCA)")
print("  df_with_pca.csv →  K-Means, Hierarchical, DBSCAN (with PCA)")
print("  y_true.csv      →  Evaluation notebook (ARI, NMI)")